# 🚀 Notebook: Full AUTSL Dataset Landmark Extraction (MediaPipe Holistic)

## 🎯 Objective
Extract **MediaPipe Holistic landmarks** (Face, Left Hand, Right Hand, Pose) for the **ENTIRE AUTSL Dataset**.
This notebook is designed for **reliability, long-running execution, and resume capability** on Google Colab.

### ✨ Key Features:
1.  **Full Dataset Processing**: No filtering, processes all available training, validation, and test videos.
2.  **Smart Resume (Auto-Save)**: Checks existing files in Google Drive before processing. If a video is already processed, it skips it. **You can stop and restart this notebook anytime without losing progress.**
3.  **Robust Error Handling**: Skips corrupted videos and logs them to a file for later inspection.
4.  **Direct Drive Storage**: Saves individual `.parquet` files to Google Drive immediately after processing.
5.  **Status Logging**: Maintains a simple `status.json` log to track progress.

---

In [3]:
# 📦 Install Dependencies
!pip install -q mediapipe==0.10.14 kagglehub pyarrow fastparquet

In [4]:
import cv2
import mediapipe as mp
import pandas as pd
import numpy as np
import os
import kagglehub
from tqdm.notebook import tqdm
import gc
import glob
import json
import time
from datetime import datetime
from google.colab import drive

# 📂 Mount Google Drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# ==========================================
# ⚙️ Configuration
# ==========================================

# Output Directory in Drive
OUTPUT_DIR = "/content/drive/MyDrive/AUTSL_Full_Parquet"

# MediaPipe Settings
FRAME_STRIDE = 1          # Process every frame for maximum quality
RESIZE_HEIGHT = 512       # Resize frames to 512px height (maintains aspect ratio usually, or square)

MP_CONFIG = {
    "static_image_mode": False,
    "model_complexity": 1,
    "enable_segmentation": False,
    "refine_face_landmarks": True,
    "min_detection_confidence": 0.5,
    "min_tracking_confidence": 0.5
}

# Logging Files
LOG_FILE_PATH = os.path.join(OUTPUT_DIR, "processing_status.json")
ERROR_LOG_PATH = os.path.join(OUTPUT_DIR, "error_log.csv")

print(f"📂 Output Directory: {OUTPUT_DIR}")

📂 Output Directory: /content/drive/MyDrive/AUTSL_Full_Parquet


In [6]:
# ==========================================
# 📥 Download & Prepare Dataset
# ==========================================

print("⏳ Downloading/Checking AUTSL dataset via KaggleHub...")
dataset_path = kagglehub.dataset_download("sttaseen/autsl")
print(f"✅ Dataset Path: {dataset_path}")

def load_dataset_metadata(base_path):
    # Load all CSVs
    # CSV columns: [filename, label]
    df_train = pd.read_csv(os.path.join(base_path, "train_labels.csv"), header=None, names=['filename', 'label'])
    df_val = pd.read_csv(os.path.join(base_path, "val_labels.csv"), header=None, names=['filename', 'label'])
    df_test = pd.read_csv(os.path.join(base_path, "test_labels.csv"), header=None, names=['filename', 'label'])

    # Add subset column
    df_train['subset'] = 'train'
    df_val['subset'] = 'val'
    df_test['subset'] = 'test'

    # Combine all
    df_all = pd.concat([df_train, df_val, df_test], ignore_index=True)

    # Function to find video path
    def find_video_path(row):
        subset_dir = os.path.join(base_path, row['subset'])
        fname = row['filename']
        
        # Check common extensions
        candidates = [
            os.path.join(subset_dir, fname + ".mp4"),
            os.path.join(subset_dir, fname + "_color.mp4")
        ]
        
        for p in candidates:
            if os.path.exists(p):
                return p
        return None

    print("🔍 Locating video files...")
    tqdm.pandas(desc="Locating Paths")
    df_all['path'] = df_all.progress_apply(find_video_path, axis=1)

    # Filter missing
    missing_count = df_all['path'].isna().sum()
    if missing_count > 0:
        print(f"⚠️ Warning: {missing_count} videos not found on disk. Removing them from list.")
        df_all = df_all.dropna(subset=['path'])
    
    return df_all

# Check paths
train_labels_path = os.path.join(dataset_path, "train_labels.csv")
if not os.path.exists(train_labels_path):
    # Search recursively if structure is different
    hits = glob.glob(os.path.join(dataset_path, "**", "train_labels.csv"), recursive=True)
    if hits:
        base_dir = os.path.dirname(hits[0])
    else:
        raise FileNotFoundError("Could not find train_labels.csv in downloaded dataset")
else:
    base_dir = dataset_path

df_dataset = load_dataset_metadata(base_dir)
print(f"📊 Total videos to process: {len(df_dataset)}")
df_dataset.head()

⏳ Downloading/Checking AUTSL dataset via KaggleHub...


100%|██████████| 12.3G/12.3G [01:52<00:00, 118MB/s] 

Extracting files...


✅ Dataset Path: /root/.cache/kagglehub/datasets/sttaseen/autsl/versions/1
🔍 Locating video files...


Locating Paths:   0%|          | 0/36302 [00:00<?, ?it/s]

📊 Total videos to process: 36302


,filename,label,subset,path
0,signer0_sample1,41,train,/root/.cache/kagglehub/datasets/sttaseen/autsl...
1,signer0_sample2,104,train,/root/.cache/kagglehub/datasets/sttaseen/autsl...
2,signer0_sample3,205,train,/root/.cache/kagglehub/datasets/sttaseen/autsl...
3,signer0_sample4,26,train,/root/.cache/kagglehub/datasets/sttaseen/autsl...
4,signer0_sample5,191,train,/root/.cache/kagglehub/datasets/sttaseen/autsl...


In [7]:
# ==========================================
# 🧠 Extraction Core Function
# ==========================================

def process_video_mediapipe(video_path, holistic_model):
    cap = cv2.VideoCapture(video_path)
    video_landmarks = []
    frame_idx = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % FRAME_STRIDE == 0:
            # Resize to ensure consistency and speed
            h, w = frame.shape[:2]
            if h != RESIZE_HEIGHT:
                 frame = cv2.resize(frame, (RESIZE_HEIGHT, RESIZE_HEIGHT))

            # Process
            frame.flags.writeable = False
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = holistic_model.process(frame_rgb)

            frame_data = {'frame': frame_idx}

            # Helper to extract flat array or NaN
            def get_lms(res_lms):
                if res_lms:
                    return np.array([[lm.x, lm.y, lm.z] for lm in res_lms.landmark], dtype=np.float16).flatten()
                return np.nan

            frame_data['pose'] = get_lms(results.pose_landmarks)
            frame_data['left_hand'] = get_lms(results.left_hand_landmarks)
            frame_data['right_hand'] = get_lms(results.right_hand_landmarks)
            frame_data['face'] = get_lms(results.face_landmarks)

            video_landmarks.append(frame_data)

        frame_idx += 1

    cap.release()
    return video_landmarks

In [8]:
# ==========================================
# 🔄 Smart Resume & Logging Setup (Updated)
# ==========================================

import os
import glob
import json
import time
from datetime import datetime

# Ensure output directories exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. ملف السجل التاريخي (CSV) - لن يتم حذفه، سيتم الإضافة عليه دائماً (Append Mode)
HISTORY_LOG_PATH = os.path.join(OUTPUT_DIR, "extraction_history.csv")
# 2. ملف الحالة الحالية (JSON) - يتم تحديثه باستمرار ليقرأ منه الكود عند استكمال العمل
LOG_FILE_PATH = os.path.join(OUTPUT_DIR, "processing_status.json")
ERROR_LOG_PATH = os.path.join(OUTPUT_DIR, "error_log.csv")

# تهيئة ملف التاريخ مع العناوين إذا لم يكن موجوداً
if not os.path.exists(HISTORY_LOG_PATH):
    with open(HISTORY_LOG_PATH, "w") as f:
        f.write("timestamp,processed_count,last_video_id,total_elapsed_seconds\n")

# تهيئة ملف الأخطاء إذا لم يكن موجوداً
if not os.path.exists(ERROR_LOG_PATH):
    with open(ERROR_LOG_PATH, "w") as f:
        f.write("video_id,error_message\n")

def get_processed_files(output_dir):
    # مسح المجلد للبحث عن الملفات المحفوظة فعلياً
    print("📂 Scanning Drive for existing files (this may take a moment)...")
    files = glob.glob(os.path.join(output_dir, "**", "*.parquet"), recursive=True)
    # استخراج اسم الفيديو فقط بدون المسار والامتداد
    processed_ids = set(os.path.basename(f).replace('.parquet', '') for f in files)
    print(f"✅ Found {len(processed_ids)} already processed videos.")
    return processed_ids

def update_log(processed_count, last_id, start_time):
    elapsed = time.time() - start_time
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    status = {
        "last_updated": timestamp,
        "processed_count": processed_count,
        "last_video_id": last_id,
        "elapsed_seconds": round(elapsed, 2)
    }
    
    # 1. تحديث ملف JSON (للاستئناف التلقائي)
    with open(LOG_FILE_PATH, "w") as f:
        json.dump(status, f, indent=4)
        
    # 2. تحديث ملف CSV (للسجل التاريخي والدراسة اليدوية)
    with open(HISTORY_LOG_PATH, "a") as f:
        f.write(f"{timestamp},{processed_count},{last_id},{elapsed:.2f}\n")

def log_error(vid_id, error_msg):
    # تسجيل الأخطاء في ملف منفصل
    error_msg_clean = str(error_msg).replace(',', ';').replace('\n', ' ')
    with open(ERROR_LOG_PATH, "a") as f:
        f.write(f"{vid_id},{error_msg_clean}\n")

# تصفية القائمة بناءً على ما تم إنجازه
processed_ids = get_processed_files(OUTPUT_DIR)
df_remaining = df_dataset[~df_dataset['filename'].isin(processed_ids)].copy()

print(f"📉 Videos remaining: {len(df_remaining)} / {len(df_dataset)}")

📂 Scanning Drive for existing files (this may take a moment)...
✅ Found 14136 already processed videos.
📉 Videos remaining: 22166 / 36302


In [ ]:
# ==========================================
# ▶️ Main Execution Loop (With Time Resume)
# ==========================================

mp_holistic = mp.solutions.holistic

# ------------------------------------------
# ⏱️ منطق استكمال الوقت (Time Resume Logic)
# ------------------------------------------
previous_elapsed = 0
# نحاول قراءة الوقت السابق فقط إذا كانت هناك ملفات تمت معالجتها بالفعل
if len(processed_ids) > 0 and os.path.exists(LOG_FILE_PATH):
    try:
        with open(LOG_FILE_PATH, 'r') as f:
            status = json.load(f)
            previous_elapsed = status.get('elapsed_seconds', 0)
        print(f"⏱️ Resuming timer from previous session: {previous_elapsed/3600:.2f} hours")
    except Exception as e:
        print(f"⚠️ Could not load previous time: {e}")
        pass

# ضبط وقت البداية الحالي ليعكس الوقت الإجمالي (السابق + الحالي)
start_time = time.time() - previous_elapsed
current_processed_count = len(processed_ids)

print("🚀 Starting Extraction Loop...")
print("💾 Progress is saved immediately. History log updated every 5 videos.")

with mp_holistic.Holistic(**MP_CONFIG) as holistic:
    # التكرار فقط على الفيديوهات المتبقية
    for idx, row in tqdm(df_remaining.iterrows(), total=len(df_remaining), desc="Processing"):
        vid_id = row['filename']
        subset = row['subset']
        label = row['label']
        video_path = row['path']

        # تجهيز مسار الحفظ
        save_dir = os.path.join(OUTPUT_DIR, subset, str(label))
        os.makedirs(save_dir, exist_ok=True)
        save_path = os.path.join(save_dir, f"{vid_id}.parquet")

        # فحص إضافي: إذا كان الملف موجوداً (لتجنب التكرار في حال إعادة التشغيل السريع)
        if os.path.exists(save_path):
            continue

        try:
            # استخراج النقاط
            landmarks_list = process_video_mediapipe(video_path, holistic)
            
            if landmarks_list:
                # تحويل لقاعدة بيانات وحفظها كملف Parquet
                df_vid = pd.DataFrame(landmarks_list)
                df_vid.to_parquet(save_path, engine='pyarrow', compression='snappy')
                
                current_processed_count += 1
                
                # تحديث السجلات كل 5 فيديوهات لضمان دقة البيانات
                if current_processed_count % 5 == 0:
                    update_log(current_processed_count, vid_id, start_time)
            
            else:
                # إذا كان الفيديو تالفاً أو لم تُستخرج منه إطارات
                print(f"⚠️ Warning: No frames processed for {vid_id}")
                log_error(vid_id, "No frames processed")

        except Exception as e:
            print(f"❌ Error processing {vid_id}: {str(e)}")
            log_error(vid_id, str(e))

        # تنظيف الذاكرة دورياً
        if idx % 50 == 0:
            gc.collect()

print("🎉 COMPLETE! All videos processed.")
update_log(current_processed_count, "COMPLETED", start_time)

⏱️ Resuming timer from previous session: 25.87 hours
🚀 Starting Extraction Loop...
💾 Progress is saved immediately. History log updated every 5 videos.


Processing:   0%|          | 0/22166 [00:00<?, ?it/s]

⚠️ Warning: No frames processed for signer2_sample1161
⚠️ Warning: No frames processed for signer3_sample152
⚠️ Warning: No frames processed for signer5_sample618
⚠️ Warning: No frames processed for signer8_sample1406


/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


⚠️ Warning: No frames processed for signer15_sample276
⚠️ Warning: No frames processed for signer20_sample424
⚠️ Warning: No frames processed for signer22_sample612
